# Step 1 Setup the Persistance Layer

In [ ]:
# CELL 1

import os
import sys
import json
import uuid
from datetime import datetime

# Add parent directory to path to import from utils
sys.path.append(os.path.join(os.path.dirname(os.getcwd())))

import pymongo
from pymongo import MongoClient
import requests
import voyageai

from utils import create_index, create_search_index, check_index_ready, set_env

PASSKEY = os.getenv("PASSKEY", "replace-with-passkey")

# ----- MONGODB SETUP -----
# If you are using your own MongoDB Atlas cluster, use the connection string for your cluster here
MONGODB_URI = os.environ.get("MONGODB_URI")
# Initialize a MongoDB Python client
mongodb_client = MongoClient(MONGODB_URI, appname="devrel-workshop-agent-memory")
# Check the connection to the server
mongodb_client.admin.command("ping")

# ----- API KEY SETUP -----
# Set the URL for our AI model proxy
PROXY_ENDPOINT = os.environ.get("PROXY_ENDPOINT")
# Obtain API keys from our AI model proxy and set them as environment variables-- DO NOT CHANGE
set_env(["voyageai"], PASSKEY)

# Initialize the Voyage AI client
vo = voyageai.Client()

# Define a function to generate embeddings using the Voyage AI API
def get_embedding(text: str, input_type: str) -> list[float]:
    """
    Get embeddings for an input text

    Args:
        text (str): Text to embed
        input_type (str): One of "document" or "query"

    Returns:
        list[float]: Embedding of the input text
    """
    # Use the `embed` method of the Voyage API to embed a piece of text with the following arguments:
    # texts: A list of strings to embed, in this case `text` wrapped in a list.
    # model: `voyage-4`
    # input_type: The `input_type` passed to the `get_embedding` function, which is either "document" or "query".
    embds_obj = vo.embed(texts=[text], model="voyage-4", input_type=input_type)
    # Extract embeddings from the embeddings object (`embds_obj`)
    embeddings = embds_obj.embeddings[0]
    return embeddings


# Database name
DB_NAME = "memory"
# Name of the chat history collection
CHATS_COLLECTION_NAME = "short_term_chats"
# Name of the semantic memory collection
SEMANTIC_COLLECTION_NAME = "long_term_semantic"
# Name of the procedural memory collection
PROCEDURAL_COLLECTION_NAME = "long_term_procedural"
# Name of the vector search index
VS_INDEX_NAME = "vector_index"

# Access the `DB_NAME` database.
db = mongodb_client[DB_NAME]
# Access the `CHATS_COLLECTION_NAME` collection.
chats_collection = db[CHATS_COLLECTION_NAME]
# Access the `SEMANTIC_COLLECTION_NAME` collection.
semantic_collection = db[SEMANTIC_COLLECTION_NAME]
# Access the `PROCEDURAL_COLLECTION_NAME` collection.
procedural_collection = db[PROCEDURAL_COLLECTION_NAME]


# Seed the procedural memories collection with memories
with open(f"../data/memories.json", "r") as data_file:
    json_data = data_file.read()

data = json.loads(json_data)

print(f"Deleting existing documents from the {PROCEDURAL_COLLECTION_NAME} collection.")
procedural_collection.delete_many({})
procedural_collection.insert_many(data)
print(
    f"{procedural_collection.count_documents({})} documents ingested into the {PROCEDURAL_COLLECTION_NAME} collection."
)

# Use the `create_index` function from the `utils` module to:
# 1. create an index on the "session_id" field
# 2. create a time-to-live (TTL) index to automatically delete documents after 1 year (31536000 seconds)
create_index(chats_collection, ["session_id"], "session_index")
create_index(chats_collection, ["timestamp"], "ttl_index", expireAfterSeconds=31536000)

# Use the `create_index` function from the `utils` module to create a time-to-live (TTL) index on the `semantic_collection` collection
create_index(semantic_collection, ["timestamp"], "ttl_index", expireAfterSeconds=31536000)

# Create the vector index definition for the procedural memories collection, specifying:
# path: Path to the embeddings field
# numDimensions: Number of embedding dimensions- depends on the embedding model used
# similarity: Similarity metric. One of cosine, euclidean, dotProduct.
model = {
    "name": VS_INDEX_NAME,
    "type": "vectorSearch",
    "definition": {
        "fields": [
            {
                "type": "vector",
                "path": "embedding",
                "numDimensions": 1024,
                "similarity": "cosine",
            }
        ]
    },
}

# 🚨🚨🚨 CODE_BLOCK_1 🚨🚨🚨
# Why: The procedural memory collection needs a Vector Search index before any $vectorSearch
# query (see CODE_BLOCK_7) can run against its `embedding` field.
# Task: The call to `create_search_index` below is almost complete. Fill in just the last
# argument with the index definition you built above (`model`).
# Docs: https://www.mongodb.com/docs/atlas/atlas-vector-search/vector-search-type/#create-and-manage-an-atlas-vector-search-index
create_search_index(procedural_collection, VS_INDEX_NAME, <CODE_BLOCK_1>)


# Step 2.1 Define short-term memory functions

These functions are not exposed as agent tools because chat history needs to be persisted at fixed points in the agent loop (after user input, LLM responses, and tool results) — not based on LLM judgment. We call them directly in code. 

In [ ]:
# CELL 2.1.a

# Define a function to store chat messages to MongoDB
def create_short_term_mem(session_id: str, role: str, content: str) -> None:
    """
    Store a chat message in a MongoDB collection.

    Args:
        session_id (str): Session ID of the message.
        role (str): Role for the message. One of `user` or `assistant`.
        content (str): Content of the message.
    """
    # Create the message object to persist in MongoDB
    msg_obj = {
        "session_id": session_id,
        "role": role,
        "content": content,
        "timestamp": datetime.now(),
    }
    # 🚨🚨🚨 CODE_BLOCK_2 🚨🚨🚨
    # Why: Short-term (chat history) memory only exists once it's written to the `chats_collection`;
    # every call site in this notebook assumes messages are durably persisted here.
    # Task: Insert `msg_obj` into `chats_collection` using the single-document insert method.
    # Docs: https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.insert_one
    <CODE_BLOCK_2>

In [ ]:
# CELL 2.1.b

# Define a function to retrieve chat history from MongoDB
def retrieve_short_term_mem(session_id: str) -> list:
    """
    Retrieve chat history for a session.

    Args:
        session_id (str): Session ID to retrieve chat history for.

    Returns:
        List: List of chat messages.
    """
    # 🚨🚨🚨 CODE_BLOCK_3 🚨🚨🚨
    # Why: The agent loop (CELL 4.1.c) needs this session's prior turns, in order, to send as
    # conversation context to the LLM on every iteration.
    # Task: Query `chats_collection` for documents where `session_id` matches, projecting only
    # `role` and `content` (exclude `_id`), sorted ascending by `timestamp`. Assign the result to `cursor`.
    # Docs:
    #   find/projection: https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.find
    #   sort:            https://pymongo.readthedocs.io/en/stable/api/pymongo/cursor.html#pymongo.cursor.Cursor.sort
    cursor = <CODE_BLOCK_3>
    docs = list(cursor)
    print("Retrieving chat history...")
    print(f"Found {len(docs)} messages")
    return docs

# Step 2.2 Define long-term memory tools (functions)

In [ ]:
# CELL 2.2.a

# Define a tool to create long-term semantic memories
def create_long_term_semantic_mem(user_id: str, memories: list[str]) -> str:
    """
    Save user facts and preferences

    Args:
        user_id (str): User ID
        memories (list[str]): Facts and preferences to save

    Returns:
        str: Save message
    """
    docs = []
    # Iterate through the list of user memories to save to MongoDB
    for memory in memories:
        # Create a separate document for each memory
        doc = {
            "user_id": user_id,
            "content": memory,
            "timestamp": datetime.now()
        }
        docs.append(doc)
    # Bulk-insert `docs` into the `semantic_collection` collection using the `insert_many()` method
    semantic_collection.insert_many(docs)
    return f"Saved {len(memories)} memories"

# Define a tool to retrieve long-term semantic memories
def retrieve_long_term_semantic_mem(user_id: str) -> str:
    """
    Retrieve memories for a particular user

    Args:
        user_id (str): User ID

    Returns:
        str: Retrieved memories formatted as a string
    """
    # 🚨🚨🚨 CODE_BLOCK_4 🚨🚨🚨
    # Why: This is called at the start of every session (see the MEMORY PROTOCOL in CELL 4.1.b) so
    # the agent can recall what it already knows about the user before responding.
    # Task: Query `semantic_collection` for documents where `user_id` matches, projecting only
    # `content` and `timestamp` (exclude `_id`), sorted descending by `timestamp` (most recent first).
    # Assign the result to `cursor`.
    # Docs:
    #   find/projection: https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.find
    #   sort:            https://pymongo.readthedocs.io/en/stable/api/pymongo/cursor.html#pymongo.cursor.Cursor.sort
    cursor = <CODE_BLOCK_4>
    docs = list(cursor)
    
    if not docs:
        return f"No memories found for user {user_id}"
    
    # Convert memories into the following format:
    # - 2026-03-12 14:54:13.296000 User prefers MongoDB for data storage
    lines = [f"- {d['timestamp']} {d['content']}" for d in docs]
    return f"Memories for {user_id}:\n" + "\n".join(lines)

In [ ]:
# CELL 2.2.b

# 🚨🚨🚨 CODE_BLOCK_5 🚨🚨🚨
# Why: `write_notes_to_scratchpad` and `create_long_term_procedural_mem` (CODE_BLOCK_6) both need
# to agree on where notes live — one appends to this file, the other reads it back.
# Task: Set `NOTES_FILE` to a path for a local scratchpad file, e.g. "./NOTES.md".
# Docs: https://docs.python.org/3/library/functions.html#open
NOTES_FILE = <CODE_BLOCK_5>

# Define a tool to write notes to a local scratchpad
def write_notes_to_scratchpad(notes: list[str]) -> str:
    """
    Write notes to a local file

    Args:
        notes (list[str]): Notes to write to file

    Returns:
        str: Note recorded message
    """
    # Open the notes file in append mode and write each note to the file, prefixed with the timestamp
    with open(NOTES_FILE, "a") as f:
        for note in notes:
            f.write(f"- [{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {note}\n")
    return f"Recorded {len(notes)} notes"

In [ ]:
# CELL 2.2.c

# Define a tool to generate procedural memories from notes using an LLM, and persist them to MongoDB
def create_long_term_procedural_mem() -> str:
    """
    Generate a procedural memory from notes and save to MongoDB
    
    Returns:
        str: Procedure save acknowledgment
    """
    # Read notes from the scratchpad
    try:
        with open(NOTES_FILE, "r") as f:
            notes_content = f.read().strip()
    except FileNotFoundError:
        return "No notes found to create procedure from."
    
    if not notes_content:
        return "Notes file is empty."
    
    # Use an LLM to generate a procedure title and summary from the notes content. 
    prompt = f"""Based on these session notes, extract a reusable, step-by-step implementation guide that a coding agent could 
    follow to reproduce a similar project from scratch.

    Focus on:
    - The specific sequence of implementation steps
    - Which tools, libraries, and configurations were used and why
    - Key decisions and their rationale
    - Pitfalls encountered and how to avoid them

    Write it as instructions ("do this, then do this"), not as a narrative of what happened.

    Also provide a brief, descriptive title for the procedure.

    Notes: {notes_content}
    """
    # Format the message to the LLM in the format [{"role": <role_value>, "content": <content_value>}]
    # The `role` value for user messages must be "user"
    # Use the `prompt` created above to populate the `content` field in the chat message
    messages = [{"role": "user", "content": prompt}]
    # Create a JSON schema to get a JSON object as output from the LLM, containing only `title` and `description` fields.
    json_schema = {        
        # HINT: The output needs to be a JSON "object"        
        "type":  "object",
        # The JSON object should contain "title" and "description" fields
        "properties": {
            "title": {"type": "string", "description": "Brief, descriptive title (5-10 words)"},
            "description": {"type": "string", "description": "A summary of what was accomplished, the approach taken, and key insights (max 2 paragraphs)"},
        },
        # Specify that the "title" and "description" fields are both required
        "required":["title", "description"],
        # There should be no additional fields in the output, so we set the `additionalProperties` field to `False`
        "additionalProperties": False
    }
    # Use the AI model proxy to get an LLM response
    response = requests.post(url=PROXY_ENDPOINT, json={"task": "completion", "data": {
            "messages": messages,
            "output_config": {"format": {"type": "json_schema", "schema": json_schema}}
        }}
    )
    if response.status_code == 200:
        result = response.json()
    else:
        # If error in LLM response, print the error message
        print(response.json()["error"])
        return "Error generating procedure."
    
    # Parse the LLM response
    procedure = json.loads(result["content"][0]["text"])
        
    # Generate embeddings for the `description` field of the `procedure` using the `get_embedding` function defined above.
    # Specify the `input_type` argument as "document".
    embedding = get_embedding(procedure["description"], "document")
    # Create the procedure document to insert into MongoDB
    procedure_doc = {
        "title": procedure["title"],
        "description": procedure["description"],
        "timestamp": datetime.now(),
        "embedding": embedding
    }
    # 🚨🚨🚨 CODE_BLOCK_6 🚨🚨🚨
    # Why: The procedure isn't durably remembered across sessions until it's written to
    # `procedural_collection` — this is what `retrieve_long_term_procedural_mem` (CODE_BLOCK_7)
    # searches over later.
    # Task: Insert `procedure_doc` into `procedural_collection` using the single-document insert method.
    # Docs: https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.insert_one
    <CODE_BLOCK_6>
    
    # Clear the notes file after saving the procedure
    open(NOTES_FILE, "w").close()
    
    return f"Procedure saved: '{procedure['title']}' - Notes cleared."

In [ ]:
# CELL 2.2.d

# Define a tool to retrieve procedural memories from MongoDB using semantic search
def retrieve_long_term_procedural_mem(query: str) -> str:
    """
    Retrieve previous procedures using semantic search

    Args:
        note (str): The 

    Returns:
        str: Relevant procedures formatted as strings
    """
    # Use the `get_embedding` function defined above to generate an embedding for the input `query`.
    # Specify the `input_type` argument as "query".
    query_embedding = get_embedding(query, "query")
    # 🚨🚨🚨 CODE_BLOCK_7 🚨🚨🚨
    # Why: This is how the agent finds relevant past procedures (see the MEMORY PROTOCOL in
    # CELL 4.1.b) — plain keyword matching wouldn't catch semantically similar-but-differently-worded tasks.
    # Task: The `$vectorSearch` stage below is almost complete. Fill in just the `queryVector` field
    # with the embedding you generated above (`query_embedding`).
    # Docs: https://www.mongodb.com/docs/atlas/atlas-vector-search/vector-search-stage/
    pipeline = [
        {
            "$vectorSearch": {
                "index": VS_INDEX_NAME,
                "path": "embedding",
                "queryVector": <CODE_BLOCK_7>,
                "numCandidates": 100,
                "limit": 5
            }
        },
        {"$project": {"_id": 0, "title": 1, "description": 1}}
    ]
    # Execute the aggregation `pipeline` on the `procedural_collection` collection.
    cursor = procedural_collection.aggregate(pipeline)
    docs = list(cursor)
    
    if not docs:
        return "No relevant procedures found."

    # Format the retrieved procedures
    output = "Relevant past procedures:\n"
    for i, ep in enumerate(docs, 1):
        output += f"{i}. {ep['title']}\n"
        output += f"{ep['description']}\n\n"
    return output

# Step 3.1: Define tool schemas for the Agent

In [ ]:
# CELL 3.1

# Create JSON schemas for the tools, so the LLM knows what functions it can call, what each function does, and what arguments to provide.
memory_tools = [
    {
        "name": "create_long_term_semantic_mem",
        "description": "Save user preferences or facts as memories.",
        # Guarantee schema validation on tool names and inputs by setting `strict` to True.
        "strict": True,
        # 🚨🚨🚨 CODE_BLOCK_8 🚨🚨🚨
        # Why: This schema is what tells the LLM the exact shape of arguments to send when it decides
        # to call `create_long_term_semantic_mem` — get it wrong and tool calls will fail validation.
        # Task: Define `input_schema` as a JSON Schema object with one required property, `content`:
        #   an array of strings, described as "List of memories to save". No additional properties allowed.
        # Compare with the other tool schemas below (e.g. `write_notes_to_scratchpad`'s `notes` property)
        # for the expected shape.
        # Docs: https://json-schema.org/understanding-json-schema/reference/array
        "input_schema": <CODE_BLOCK_8>
    },
    {
        "name": "retrieve_long_term_semantic_mem",
        "description": "Retrieve stored memories for a user. Use at conversation start to get the user's profile.",
        "strict": True,
        "input_schema": {                    
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False,
        }
    },
    {
        "name": "write_notes_to_scratchpad",
        "description": "Write notes about the current task to a local file.",
        "strict": True,
        "input_schema": {
            "type": "object",
            "properties": {
                "notes": {
                    "type": "array", 
                    "items": {"type": "string"},
                    "description": "List of notes to write to file."
                },
            },
            "required": ["notes"],
            "additionalProperties": False,
        }
    },
    {
        "name": "create_long_term_procedural_mem",
        "description": "Create a procedural memory from accumulated notes.",
        "strict": True,
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False,
        }
    },
    {
        "name": "retrieve_long_term_procedural_mem",
        "description": "Search past procedural memories by semantic similarity.",
        "strict": True,
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Description of the problem or topic to search"
                },
            },
            "required": ["query"],
            "additionalProperties": False,
        }
    }
]

# Step 3.2: Define tool execution function

In [ ]:
# CELL 3.2

# Define a function to execute the tool calls
# NOTE: LLMs don't handle tool execution. They only determine which tool to execute and the arguments. 
def execute_tool(tool_name: dict, tool_args: dict, session: dict) -> str:
    """
    Coordinate tool execution

    Args:
        tool_name (str): Tool name
        tool_args (dict): Arguments for the tool call
        session (dict): Session information

    Returns:
        str: Tool output formatted as string
    """
    if tool_name == "create_long_term_semantic_mem":
        # 🚨🚨🚨 CODE_BLOCK_9 🚨🚨🚨
        # Why: This is the dispatch line that actually runs `create_long_term_semantic_mem` when the
        # LLM asks for it — without it, the tool is defined and schema'd but never invoked.
        # Task: Return the result of calling `create_long_term_semantic_mem` with the current
        # session's `user_id` and the `content` argument from `tool_args` (see CELL 3.1's schema
        # for the argument name the LLM will send).
        # Compare with the other branches below (e.g. `write_notes_to_scratchpad`) for the pattern.
        <CODE_BLOCK_9>
    elif tool_name == "retrieve_long_term_semantic_mem":
        return retrieve_long_term_semantic_mem(session["user_id"])
    elif tool_name == "write_notes_to_scratchpad":
        return write_notes_to_scratchpad(tool_args["notes"])
    elif tool_name == "create_long_term_procedural_mem":
        return create_long_term_procedural_mem()
    elif tool_name == "retrieve_long_term_procedural_mem":
        return retrieve_long_term_procedural_mem(tool_args["query"])
    else:
        return f"Unknown tool: {tool_name}"

# Step 4.1: Create a memory-augmented agent

### 4.1.a: Create a session management function

In [ ]:
# CELL 4.1.a

# Create a session object to maintain state for a user session that tracks the following:
# session_id: Unique session ID 
# user_id: Current user ID
# input_tokens: Number of input tokens used in the session
# output_tokens: Number of output tokens generated during the session
# max_tokens: The LLM's context window limit
def create_session(user_id: str, max_tokens: int=1000000) -> dict:
    """
    Create a session object to track session metadata

    Args:
        user_id (str): Unique user identifier
        max_tokens (int): The LLM's context window limit

    Returns:
        dict: Session object
    """
    # 🚨🚨🚨 CODE_BLOCK_10 🚨🚨🚨
    # Why: Every function in this notebook that reads or writes memory (short-term, semantic, or
    # procedural) takes this `session` dict as its source of truth for `session_id`.
    # Task: Fill in a unique `session_id` string below. A good option is the first 8 characters of
    # a `uuid.uuid4()` (e.g. `str(uuid.uuid4())[:8]`).
    # Docs: https://docs.python.org/3/library/uuid.html#uuid.uuid4
    return {
        "session_id": <CODE_BLOCK_10>,
        "user_id": user_id,
        "input_tokens": 0,
        "output_tokens": 0,
        "max_tokens": max_tokens
    }

### Step 4.1.b: Define a dynamic system prompt for the agent

In [ ]:
# CELL 4.1.b

# Calculate context window usage during the session-- we will use this as one of the triggers for memory creation
def get_context_usage(session: dict) -> float:
    """
    Calculate context window usage as a percentage

    Args:
        session (dict): Session object

    Returns:
        float: % of the context window used
    """
    total_tokens = session["input_tokens"] + session["output_tokens"]
    # 🚨🚨🚨 CODE_BLOCK_11 🚨🚨🚨
    # Why: The system prompt below reports this percentage back to the LLM every turn (see
    # "CURRENT SESSION STATUS"), and the MEMORY PROTOCOL explicitly tells the agent to save a
    # procedure once usage crosses 70% — this calculation is what makes that threshold meaningful.
    # Task: `total_tokens` divided by `session["max_tokens"]` gives a fraction (0-1). Fill in what
    # to multiply it by to turn that fraction into a percentage.
    return (total_tokens / session["max_tokens"]) * <CODE_BLOCK_11>

# Create a system prompt providing guidance on how and when to use the memory tools
def get_system_prompt(session: dict) -> str:
    """
    Build system prompt with context usage status

    Args:
        session (dict): Session object

    Returns:
        str: Agent's system prompt
    """
    usage_pct = get_context_usage(session)
    prompt = f"""You are a helpful coding assistant with memory capabilities.

    ## CURRENT SESSION STATUS
    - Context window usage: {usage_pct:.1f}%

    ## GENERAL INSTRUCTIONS
    - Ask follow-up questions if the user request is unclear or too broad.
    - Keep responses concise and focused. Avoid lengthy code examples unless specifically requested.

    ## MEMORY PROTOCOL
    **At conversation start:**
    - Use `retrieve_long_term_semantic_mem` to retrieve user preferences and facts.
    - Prioritize RECENT information if memories conflict.

    **When to search for past procedures (`retrieve_long_term_procedural_mem`):**
    - ANY technical or coding question
    - Procedures are YOUR internal knowledge from past sessions. Use them SILENTLY to inform your approach.

    **During the conversation:**
    - Use `write_notes_to_scratchpad` to record:
        - Implementation steps taken and their order
        - Tools, libraries, and configurations chosen (and WHY)
        - Errors encountered and how they were resolved
        - User feedback that changed the approach
    - Keep EACH NOTE to 2-3 SENTENCES.
    - Write notes after each meaningful implementation step, not just at the end.

    **When you learn something important about the user:**
    - Use `create_long_term_semantic_mem` to save facts and preferences such as preferred language, frameworks, coding style, and experience level.
    - Keep EACH MEMORY to 1 SENTENCE.

    **When to save a procedure (`create_long_term_procedural_mem`):**
    - A multi-step implementation task is COMPLETE — not just a single question answered or concept explained.
    - The solution involved specific tools, configurations, or a sequence of steps that would be useful to reproduce.
    - Do NOT save procedures for simple factual Q&A, single function explanations, or debugging one-liners.
    - Write the procedure as INSTRUCTIONS a future agent could follow ("Set up X, then configure Y"), not as a summary of what happened ("We set up X, then configured Y").
    - Include pitfalls encountered and how to avoid them.
    - DEFINITELY save a procedure when the context window usage exceeds 70%.

    ASSUME INTERRUPTION: Your context window might be reset at any moment, so you risk losing any progress that is not recorded in your notes.
    """
    
    return prompt

### Step 4.1.c: Define the agent loop

In [ ]:
# CELL 4.1.c

# Define the agent execution loop
def chat(user_input: str, session: dict) -> None:
    """
    The agent execution loop

    Args:
        user_input (str): The user's question
        session (dict): Session object
    """
    session_id = session["session_id"]
    print("===== Human message =====")
    print(user_input)
    # Use the `create_short_term_mem` function to add the `user_input` to the chat history for this session
    # The `role` value for user messages is "user"
    create_short_term_mem(session_id, "user", user_input)
    # Run the agent loop until the LLM decides it has the final answer
    while True:
        # Use the `get_system_prompt` function to build a dynamic system prompt with the current context usage status
        system_prompt = get_system_prompt(session)

        # Use the `retrieve_short_term_mem` function to retrieve chat messages for the current session from MongoDB
        messages = retrieve_short_term_mem(session_id)
        
        # 🚨🚨🚨 CODE_BLOCK_12 🚨🚨🚨
        # Why: This is the actual call to the LLM — everything else in the loop (system prompt,
        # chat history, tool schemas) only matters because it gets sent here on every iteration.
        # Task: Call `requests.post`, with `url=PROXY_ENDPOINT` and a JSON body of the form
        # {"task": "completion", "data": {...}}, where "data" carries:
        #   "system": system_prompt, "messages": messages, "tools": memory_tools
        # Assign the result to `response`.
        # Docs: https://requests.readthedocs.io/en/latest/api/#requests.post
        response = <CODE_BLOCK_12>
        if response.status_code == 200:
            result = response.json()
        else:
            # If error in LLM response, print the error message
            print(response.json()["error"])
            return
        
        # Update the session's token usage
        session["input_tokens"] += result["input_tokens"]
        session["output_tokens"] += result["output_tokens"]
        
        llm_response = result["content"]
        print("===== AI message =====")
        text_parts = [block["text"] for block in llm_response if block.get("type") == "text"]
        # Print the text blocks in the LLM's response
        if text_parts:
            print("\n".join(text_parts))
        # Use the `create_short_term_mem` function to store the LLM response (`llm_response`) in the `chats` collection
        # The `role` value for LLM responses is "assistant"
        create_short_term_mem(session_id, "assistant", llm_response)
        
        # Final answer reached- break out of the loop
        if result["stop_reason"] == "end_turn":
            break

        # Maximum output token limit reached. Stop the loop to avoid continuing with a truncated response.
        elif result["stop_reason"] == "max_tokens":
            print("The model reached the maximum output token limit before finishing. Try asking a narrower question.")
            break
  
        # Handle tool calls- there can be multiple tool calls in the LLM's response
        elif result["stop_reason"] == "tool_use":
            tool_outputs = []
            for block in llm_response:
                if block.get("type") == "tool_use":
                    # Get the tool name from the LLM response
                    tool_name = block["name"]
                    # Get the tool arguments from the LLM response
                    tool_args = block["input"]
                    print("===== Tool Call =====")
                    print(f"Calling {tool_name} with args: {tool_args}")
                    
                    # Use the `execute_tool` function defined in Step 7 to execute the right tool
                    # Use the`tool_name` and `tool_args` identified by the LLM, and the `session` object as parameters
                    tool_output = execute_tool(tool_name, tool_args, session)
                    print(f"===== Tool Outcome: {tool_name} =====")
                    print(tool_output)
                    
                    # Collect all tool outputs
                    tool_outputs.append({
                        "type": "tool_result",
                        "tool_use_id": block["id"],
                        "content": tool_output
                    })
            # Use the `create_short_term_mem` function to store the `tool_outputs` in the `chats` collection
            # The `role` value for tool outputs is "user"
            create_short_term_mem(session_id, "user", tool_outputs)

# Step 5: Test you Agent's memory

In [ ]:
# Create a new session
session_1 = create_session(user_id="mdb_user")
print(f"Started session: {session_1['session_id']}")

In [ ]:
# Test 1: Ask a coding question
# The agent should take notes and search for relevant past procedures
response = chat("Help me create an AI-powered shopping assistant.", session_1)

In [ ]:
# Test 2: Tell it some of your personal preferences
# The agent should extract and persist semantic memories 
response = chat("MongoDB is my preferred database and Voyage AI is my preferred embedding provider.", session_1)

In [ ]:
# Test 3: Check cross-session behavior
# The agent should recall long-term memories from previous sessions; chat history should get reset
session_2 = create_session(user_id="mdb_user")
print(f"Started session: {session_2['session_id']}")

response = chat("What do you know about me?", session_2)